In [ ]:
import numpy as np
import random
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

print("X shape ", X.shape)
print("y shape ", y.shape)

print("X_train ", X_train.shape)
print("y_train ", y_train.shape)

print("X_test ", X_test.shape)
print("y_test ", y_test.shape)

### Linear Regression

In [ ]:
reg = LinearRegression()
reg.fit(X_train,y_train)

In [ ]:
print("Co-efficeint ", reg.coef_)
print("Intercept ", reg.intercept_)

In [ ]:
y_pred = reg.predict(X_test)
r2_score(y_test,y_pred)

### Always scale before gradient descent!

In [ ]:
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"X_train: {X_train_sc.shape}")   # (353, 10)
print(f"X_test:  {X_test_sc.shape}")    # (89, 10)
print(f"y_train: {y_train.shape}")

### MBGDRegressor


In [ ]:
class MBGDRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, batch_size=32, lr=0.01, epochs=100):
        self.batch_size = batch_size
        self.lr         = lr
        self.epochs     = epochs
        self.loss_hist  = []

    def fit(self, X, y):
        self.intercept_ = 0
        self.coef_      = np.ones(X.shape[1])
        n               = X.shape[0]

        for epoch in range(self.epochs):
            # --- production-style: shuffle ONCE, then slice ---
            idx = np.random.permutation(n)
            for start in range(0, n, self.batch_size):
                batch  = idx[start : start + self.batch_size]
                Xb, yb = X[batch], y[batch]

                m = Xb.shape[0]

                y_hat = np.dot(Xb, self.coef_) + self.intercept_
                err   = yb - y_hat

                self.intercept_ -= self.lr * (-2 * np.mean(err))
                self.coef_      -= self.lr * (-2 * np.dot(Xb.T, err) / m)

            # track full-dataset loss after each epoch
            full_hat = np.dot(X, self.coef_) + self.intercept_
            self.loss_hist.append(np.mean((y - full_hat)**2))

        return self

    def predict(self, X):
        return np.dot(X, self.coef_) + self.intercept_

In [ ]:
batch_size=32
model = MBGDRegressor(batch_size=24, lr=0.010, epochs=100)
model.fit(X_train_sc, y_train)

y_pred = model.predict(X_test_sc)
print(f"R² : {r2_score(y_test, y_pred):.4f}")
print(f"updates/epoch : {len(range(0, 353, batch_size))}")

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mbr", MBGDRegressor(batch_size=32, lr=0.01, epochs=100)),
])

cv_scores = cross_val_score(pipe, X, y, cv=5, scoring="r2")
print(f"CV R² scores : {cv_scores.round(4)}")
print(f"Mean CV R²   : {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")